# PyTorch from Scratch: A Toy Classification Example

We will builds, trains, and evaluates a small neural network in PyTorch,
step by step. The goal isn't to solve a hard problem, it's to show every
core PyTorch building block in one short, runnable example:

1. **Tensors** : the basic data structure
2. **`nn.Module`** : how you define a network
3. **A loss function** : measures how wrong the model is
4. **An optimizer** : updates the weights
5. **The training loop** : forward → loss → backward → step
6. **`torch.no_grad()`** : running inference without tracking gradients

### The toy problem
We'll classify 2D points arranged as **two concentric circles** (an inner
ring = class 0, an outer ring = class 1).

## 1. Imports and setup

We fix the random seeds so the notebook produces the same results every time

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

In [ ]:
torch.manual_seed(0)
np.random.seed(0)

## 2. Creating the toy dataset

We generate
- two rings of points using polar coordinates (radius + angle),
- add a little Gaussian noise, and
- label the inner ring `0` and the outer ring `1`.

This part uses plain `NumPy`


In [ ]:
def make_circles(n_samples=200, noise=0.1):
    n = n_samples // 2
    theta1 = np.random.uniform(0, 2 * np.pi, n)
    theta2 = np.random.uniform(0, 2 * np.pi, n)
    r1 = 1.0 + np.random.randn(n) * noise      # inner circle -> class 0
    r2 = 2.5 + np.random.randn(n) * noise      # outer circle -> class 1

    x1 = np.stack([r1 * np.cos(theta1), r1 * np.sin(theta1)], axis=1)
    x2 = np.stack([r2 * np.cos(theta2), r2 * np.sin(theta2)], axis=1)

    X = np.vstack([x1, x2]).astype(np.float32)
    y = np.concatenate([np.zeros(n), np.ones(n)]).astype(np.float32)
    return X, y

n_samples = 1000
noise = 0.3
X, y = make_circles(n_samples, noise)
print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
plt.figure(figsize=(5, 5))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap="RdBu", edgecolors="k")
plt.title("Toy dataset: two concentric circles")
plt.xlabel("x1")
plt.ylabel("x2")
plt.show()

## 3. Tensors

PyTorch models only accept `torch.Tensor` inputs- not NumPy arrays and
not Python lists.

`torch.from_numpy()` converts a NumPy array to a tensor
that **shares the same memory** (no copy).

Note the shapes:
- `X_tensor` is `(N, 2)` - N samples, 2 features each.
- `y_tensor` is reshaped to `(N, 1)` with `.unsqueeze(1)`, because the loss
  function we'll use expects the target to have the same shape as the
  model's output (`(N, 1)`), not a flat `(N,)` vector.


In [ ]:
X_tensor = torch.from_numpy(X)                  # shape: (N, 2)
y_tensor = torch.from_numpy(y).unsqueeze(1)     # shape: (N, 1)

print(X_tensor.shape, X_tensor.dtype)
print(y_tensor.shape, y_tensor.dtype)

## 4. Defining the network with `nn.Module`

Every PyTorch model subclasses `nn.Module` and implements `forward()`,
which describes how input data flows through the layers.

We never call `forward()` directly: we call the model itself (`model(x)`), which handles some bookkeeping and then calls `forward()` for us.

Architecture of Network:

 `2 → 16 → 16 → 1`, with `Sigmoid` nonlinearities in between.

**Why the Sigmoid matter:** stacking `Linear` layers with *no* nonlinearity
between them collapses into a single linear transformation, no matter how
many layers you add.

`Sigmoid` (or any nonlinearity) is what lets the network
learn a curved decision boundary.

The output layer has 1 unit and **no activation function** — it outputs a
raw score ("logit"), not a probability. We'll turn that into a probability
with `sigmoid` later, at evaluation time.


In [ ]:
class SimpleNN(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Sigmoid(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Sigmoid(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        return self.net(x)   # raw logits, not probabilities

model = SimpleNN()
print(model)

## 5. Loss function and optimizer

- **`nn.BCEWithLogitsLoss`** combines a `sigmoid` and binary cross-entropy
  into a single, numerically stable operation. It's the standard choice
  for binary classification when your model outputs raw logits (as ours
  does).
- **`torch.optim.Adam`** is the optimizer: it looks at the gradients
  PyTorch computes for every parameter and decides how to update them.
  `model.parameters()` hands the optimizer every learnable weight and
  bias in the network.
- **`lr=0.01`** is the learning rate,  how big a step the optimizer takes on each update. Too high and training diverges; too low and it crawls.


In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

## 6. The training loop

This four-step pattern is the heart of every PyTorch training loop, and
it's worth memorizing:

1. **`optimizer.zero_grad()`** :clear gradients left over from the
   previous step (PyTorch accumulates gradients by default, so we must
   reset them each iteration).
2. **`outputs = model(X_tensor)`** : the forward pass.
3. **`loss.backward()`** : the backward pass. This is **autograd**:
   PyTorch automatically computes the gradient of the loss with respect
   to every parameter in the model.
4. **`optimizer.step()`** : the optimizer uses those gradients to update
   the weights.

We run this loop 200 times ("epochs"), and record the loss each time so we can plot how training progressed.


In [ ]:
n_epochs = 200
losses = []

for epoch in range(n_epochs):
    optimizer.zero_grad()          # 1. reset gradients
    outputs = model(X_tensor)      # 2. forward pass
    loss = criterion(outputs, y_tensor)
    loss.backward()                # 3. backward pass (autograd)
    optimizer.step()               # 4. update weights

    losses.append(loss.item())
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d}/{n_epochs} | Loss: {loss.item():.4f}")

## 7. Evaluation

For inference we wrap the code in **`torch.no_grad()`**. This tells
PyTorch not to track operations for gradient computation, which saves
memory and computation — we don't need gradients unless we're training.

The model's raw outputs are logits, so we apply `torch.sigmoid()` to turn
them into probabilities in `[0, 1]`, then threshold at `0.5` to get hard
class predictions.


In [ ]:
with torch.no_grad():
    predicted_probs = torch.sigmoid(model(X_tensor))
    predicted_labels = (predicted_probs > 0.5).float()
    accuracy = (predicted_labels == y_tensor).float().mean()

print(f"Final training accuracy: {accuracy.item() * 100:.2f}%")

## 8. Visualizing what the network learned

Two plots:

- **Left:** the training loss over time — it should drop sharply and then
  flatten out as the network converges.
- **Right:** the learned decision boundary. We evaluate the model on a
  dense grid of points covering the whole plane and color each point by
  the model's predicted probability. If training worked, you should see a
  **ring-shaped** boundary — something a linear model could never produce.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(losses)
axes[0].set_title("Training Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("BCE Loss")

xx, yy = np.meshgrid(np.linspace(-4, 4, 200), np.linspace(-4, 4, 200))
grid = np.c_[xx.ravel(), yy.ravel()].astype(np.float32)
with torch.no_grad():
    grid_probs = torch.sigmoid(model(torch.from_numpy(grid))).numpy().reshape(xx.shape)

axes[1].contourf(xx, yy, grid_probs, levels=50, cmap="RdBu", alpha=0.7)
axes[1].scatter(X[:, 0], X[:, 1], c=y, cmap="RdBu", edgecolors="k")
axes[1].set_title("Learned Decision Boundary")

plt.tight_layout()
plt.show()


## Things to try next

We can try the following

- **Remove both `ReLU` layers.** The model becomes a stack of linear
  layers with nothing in between, which collapses to a single linear
  transformation. Watch the decision boundary go from a ring back to
  something close to a straight line, and watch accuracy drop.
- **Shrink `hidden_dim` to 2.** Not enough capacity to represent the ring
  well — the boundary will look coarse or fail to converge.
- **Change the learning rate** (`lr=0.5` vs `lr=0.001`). Too high can
  cause the loss to oscillate or diverge; too low makes training crawl.
- **Swap `Adam` for `torch.optim.SGD`.** Compare how many epochs it takes
  to reach the same accuracy.
- **Add more noise** in `make_circles(noise=0.3)`. See how the boundary
  and final accuracy change when the classes overlap.
